# 02 - Feature Engineering

**Dynamic Pricing Engine** - Feature Pipeline Documentation

This notebook documents each engineered feature, its distribution, and correlation to price.

## Feature Categories
1. Temporal features (8 features)
2. Weather features (6 features)
3. Holiday features (4 features)
4. Location features (1 feature)
5. Demand proxy features (3 features)
6. Competitor pricing features (5 features)
7. Listing features (5 features)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.utils.config import load_config, PROJECT_ROOT

pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid')
config = load_config()

## Load Feature-Engineered Data

If you haven't run the pipeline yet, run:
```bash
python -m src.data.feature_engineering
```

In [ ]:
processed_dir = PROJECT_ROOT / config['data']['processed_dir']

listings = pd.read_parquet(processed_dir / 'listings_features.parquet')
calendar = pd.read_parquet(processed_dir / 'calendar_features.parquet')

print(f'Listings: {listings.shape}')
print(f'Calendar: {calendar.shape}')
print(f'\nListings columns: {listings.columns.tolist()}')
print(f'\nCalendar columns: {calendar.columns.tolist()}')

## 1. Temporal Features

In [ ]:
temporal_cols = ['day_of_week', 'day_of_month', 'week_of_year', 'month',
                 'quarter', 'is_weekend', 'season']
available_temporal = [c for c in temporal_cols if c in calendar.columns]

print('Temporal features present:', available_temporal)
print(calendar[available_temporal].describe())

# Booking rate by day of week
if 'was_booked' in calendar.columns and 'day_of_week' in calendar.columns:
    dow_booking = calendar.groupby('day_of_week')['was_booked'].mean()
    fig, ax = plt.subplots(figsize=(8, 4))
    dow_booking.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Booking Rate by Day of Week (0=Mon, 6=Sun)')
    ax.set_ylabel('Booking Rate')
    plt.tight_layout()
    plt.show()

## 2. Weather Features

In [ ]:
weather_cols = ['temperature_mean', 'precipitation_sum', 'wind_speed_max',
                'is_hot_day', 'is_cold_day', 'is_rainy_day']
available_weather = [c for c in weather_cols if c in calendar.columns]

if available_weather:
    print('Weather features:', available_weather)
    print(calendar[available_weather].describe())
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col in zip(axes, ['temperature_mean', 'precipitation_sum', 'wind_speed_max']):
        if col in calendar.columns:
            calendar[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
            ax.set_title(col)
    plt.tight_layout()
    plt.show()
else:
    print('No weather features found. Run the pipeline with internet access.')

## 3. Holiday Features

In [ ]:
holiday_cols = ['is_holiday', 'days_until_holiday', 'days_since_holiday', 'near_holiday']
available_holiday = [c for c in holiday_cols if c in calendar.columns]

if available_holiday:
    print('Holiday features:', available_holiday)
    print(calendar[available_holiday].describe())
    
    if 'is_holiday' in calendar.columns and 'was_booked' in calendar.columns:
        holiday_effect = calendar.groupby('is_holiday')['was_booked'].mean()
        print(f'\nBooking rate on holidays: {holiday_effect.get(1, 0):.3f}')
        print(f'Booking rate on non-holidays: {holiday_effect.get(0, 0):.3f}')
else:
    print('No holiday features found.')

## 4. Location Clusters

In [ ]:
if 'location_cluster' in listings.columns:
    print(f'Number of clusters: {listings["location_cluster"].nunique()}')
    print(listings.groupby('location_cluster')['price'].agg(['count', 'mean', 'median']))
    
    fig = px.scatter_mapbox(
        listings.sample(min(3000, len(listings)), random_state=42),
        lat='latitude', lon='longitude',
        color='location_cluster',
        title='Location Clusters (KMeans)',
        mapbox_style='carto-positron',
        zoom=10, height=500,
        color_continuous_scale='Set1',
    )
    fig.show()
else:
    print('No location clusters found.')

## 5. Demand Proxy Features

In [ ]:
demand_cols = ['occupancy_rate', 'reviews_per_month']
available_demand = [c for c in demand_cols if c in listings.columns]

if available_demand:
    print('Demand features:', available_demand)
    print(listings[available_demand].describe())
    
    fig, axes = plt.subplots(1, len(available_demand), figsize=(6*len(available_demand), 4))
    if len(available_demand) == 1:
        axes = [axes]
    for ax, col in zip(axes, available_demand):
        listings[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

## 6. Competitor Pricing Features

In [ ]:
comp_cols = ['median_neighborhood_price', 'mean_neighborhood_price',
             'price_vs_neighborhood', 'price_rank_in_neighborhood']
available_comp = [c for c in comp_cols if c in listings.columns]

if available_comp:
    print('Competitor features:', available_comp)
    print(listings[available_comp].describe())
    
    if 'price_vs_neighborhood' in listings.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        listings['price_vs_neighborhood'].clip(0, 3).hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
        ax.axvline(1.0, color='red', linestyle='--', label='Market Average')
        ax.set_title('Price vs Neighborhood Median (1.0 = at market)')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 7. Full Feature Correlation Matrix

In [ ]:
# Correlation of all numeric features with price
numeric_listings = listings.select_dtypes(include=[np.number])

if 'price' in numeric_listings.columns:
    price_corr = numeric_listings.corr()['price'].sort_values(ascending=False)
    print('Top 15 features correlated with price:')
    print(price_corr.head(15))
    print('\nBottom 5:')
    print(price_corr.tail(5))
    
    # Plot top correlations
    top_features = price_corr.drop('price').head(10)
    fig, ax = plt.subplots(figsize=(10, 5))
    top_features.plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Top 10 Features Correlated with Price')
    ax.set_xlabel('Correlation')
    plt.tight_layout()
    plt.show()

## Summary

| Category | Features | Key Insight |
|----------|----------|-------------|
| Temporal | 8 | Weekend premium, seasonal patterns |
| Weather | 6 | Temperature and rain affect demand |
| Holiday | 4 | Holiday proximity boosts bookings |
| Location | 1 | KMeans clusters capture area pricing |
| Demand | 3 | Review velocity = demand signal |
| Competitor | 5 | Neighborhood median anchors pricing |
| Listing | 5 | Room type, beds, amenities matter |
| **Total** | **32** | Ready for ML training |